In [1]:
# ==========================================
# PART 1: MACHINE LEARNING PIPELINE (2.1 & 2.2)
# Run this entirely inside Jupyter Notebook
# ==========================================

import pandas as pd
import numpy as np
import joblib
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

print("--- Loading Data ---")
# We use a built-in dataset to simulate your Question 1 data. 
# We will classify 'space' (simulating AI) vs 'religion' (simulating Classical) abstracts
categories = ['sci.space', 'alt.atheism']
data = fetch_20newsgroups(subset='all', categories=categories, shuffle=True, random_state=42, remove=('headers', 'footers', 'quotes'))

# Create DataFrame
df = pd.DataFrame({'abstract': data.data, 'label': data.target})
# Mapping: 1 = sci.space (AI), 0 = alt.atheism (Classical)
print(f"Dataset loaded with {len(df)} records.")

# Split Data
X_train, X_test, y_train, y_test = train_test_split(df['abstract'], df['label'], test_size=0.2, random_state=42)

print("\n--- 2.1 Training Multiple Models ---")
# Define 4 Models (Baseline Naive Bayes + 3 Advanced Models)
models = {
    "Baseline (Naive Bayes)": MultinomialNB(),
    "Linear SVC": LinearSVC(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

results = {}
for name, model in models.items():
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(stop_words='english', max_features=5000)),
        ('clf', model)
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f"{name} Accuracy: {acc:.4f}")

best_model_name = max(results, key=results.get)
print(f"\nBest Base Model: {best_model_name}")

print("\n--- 2.2 Performance Enhancement (GridSearchCV) ---")
# We will tune the LinearSVC (usually the best for text)
pipeline_tune = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('clf', LinearSVC(random_state=42))
])

# Hyperparameter grid
param_grid = {
    'tfidf__max_features': [2000, 5000, 10000],
    'clf__C': [0.1, 1, 10] 
}

print("Running Grid Search... (This may take a minute)")
grid_search = GridSearchCV(pipeline_tune, param_grid, cv=3, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Best Parameters found: {grid_search.best_params_}")
print(f"Best Cross-Validation Accuracy: {grid_search.best_score_:.4f}")

# Save the best model for the Web App
best_tuned_model = grid_search.best_estimator_
joblib.dump(best_tuned_model, 'best_classifier_model.pkl')
print("\nSUCCESS: Model saved as 'best_classifier_model.pkl' in your current folder.")
print("Now, copy the code from PART 2 into a separate 'app.py' file and run it in your terminal.")

--- Loading Data ---
Dataset loaded with 1786 records.

--- 2.1 Training Multiple Models ---
Baseline (Naive Bayes) Accuracy: 0.9413
Linear SVC Accuracy: 0.9078
Random Forest Accuracy: 0.8771


C:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:200: UserWarning: [21:02:47] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost Accuracy: 0.8436

Best Base Model: Baseline (Naive Bayes)

--- 2.2 Performance Enhancement (GridSearchCV) ---
Running Grid Search... (This may take a minute)
Best Parameters found: {'clf__C': 1, 'tfidf__max_features': 10000}
Best Cross-Validation Accuracy: 0.9146

SUCCESS: Model saved as 'best_classifier_model.pkl' in your current folder.
Now, copy the code from PART 2 into a separate 'app.py' file and run it in your terminal.
